In [5]:
import pandas as pd
import requests

# 1. Recharger le dataset original
url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv"
df = pd.read_csv(url)

# 2. Garder uniquement les colonnes qui correspondent au formulaire
colonnes_utiles = [
    'lead_time',                    # Date début - aujourd'hui
    'stays_in_weekend_nights',      # pour calculer duree_sejour
    'stays_in_week_nights',         # pour calculer duree_sejour
    'adults',                       # nb_personnes
    'children',                     # nb_personnes
    'arrival_date_year',            # pour météo
    'arrival_date_month',           # pour météo
    'arrival_date_day_of_month',    # pour météo
    'deposit_type',                 # paiement
    'is_repeated_guest',            # historique user
    'is_canceled'                   # target
]

df = df[colonnes_utiles].dropna()

# 3. Colonnes combinées
df['duree_sejour'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df['nb_personnes'] = df['adults'] + df['children'].fillna(0)
df = df.drop(['stays_in_weekend_nights', 'stays_in_week_nights',
              'adults', 'children'], axis=1)

# 4. Ajouter météo
df['arrival_date_month_num'] = pd.to_datetime(
    df['arrival_date_month'], format='%B').dt.month

df['date_arrivee'] = pd.to_datetime({
    'year':  df['arrival_date_year'],
    'month': df['arrival_date_month_num'],
    'day':   df['arrival_date_day_of_month']
})

dates_uniques = sorted(df['date_arrivee'].dt.date.unique())

response = requests.get("https://archive-api.open-meteo.com/v1/archive", params={
    "latitude":   36.4561,
    "longitude":  10.7376,
    "start_date": str(dates_uniques[0]),
    "end_date":   str(dates_uniques[-1]),
    "daily":      "temperature_2m_mean",
    "timezone":   "Africa/Tunis"
})

data = response.json()
df_meteo = pd.DataFrame({
    'date_arrivee': pd.to_datetime(data['daily']['time']).date,
    'temperature':  data['daily']['temperature_2m_mean']
})

def categoriser_meteo(temp):
    if temp > 30:   return 2  # Chaud
    elif temp < 15: return 0  # Froid
    else:           return 1  # Normal

df_meteo['meteo'] = df_meteo['temperature'].apply(categoriser_meteo)
df['date_arrivee'] = df['date_arrivee'].dt.date
df = df.merge(df_meteo[['date_arrivee', 'meteo']], on='date_arrivee', how='left')

# 5. Supprimer colonnes de dates
df = df.drop(['arrival_date_year', 'arrival_date_month',
              'arrival_date_month_num', 'arrival_date_day_of_month',
              'date_arrivee'], axis=1)

# 6. Encoder deposit_type
df['deposit_type'] = df['deposit_type'].map({
    'No Deposit': 0,
    'Non Refund': 1,
    'Refundable': 2
})

df = df.dropna()

print("Shape final:", df.shape)
print("Colonnes:", df.columns.tolist())
print(df.head())

Shape final: (119386, 7)
Colonnes: ['lead_time', 'deposit_type', 'is_repeated_guest', 'is_canceled', 'duree_sejour', 'nb_personnes', 'meteo']
   lead_time  deposit_type  is_repeated_guest  is_canceled  duree_sejour  \
0        342             0                  0            0             0   
1        737             0                  0            0             0   
2          7             0                  0            0             1   
3         13             0                  0            0             1   
4         14             0                  0            0             2   

   nb_personnes  meteo  
0           2.0      1  
1           2.0      1  
2           1.0      1  
3           1.0      1  
4           2.0      1  


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import pickle

# 1. Separer features et target
X = df.drop('is_canceled', axis=1)
y = df['is_canceled']

# 2. Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Entrainer le modele
print("Entrainement en cours...")
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# 4. Evaluation
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Rapport de classification:")
print(classification_report(y_test, y_pred,
      target_names=['Non annule', 'Annule']))
print("ROC-AUC Score:", round(roc_auc_score(y_test, y_proba), 3))

# 5. Sauvegarder le modele
pickle.dump(model, open('model.pkl', 'wb'))
print("Modele sauvegarde : model.pkl")

# 6. Tester avec une nouvelle reservation (simulation formulaire CAMPINO)
nouvelle_reservation = {
    'lead_time':          [30],   # reservation 30 jours a l'avance
    'deposit_type':       [0],    # paiement espece
    'is_repeated_guest':  [0],    # nouveau client
    'duree_sejour':       [3],    # 3 nuits
    'nb_personnes':       [4],    # 4 personnes
    'meteo':              [2]     # Chaud
}

import pandas as pd
test = pd.DataFrame(nouvelle_reservation)
proba = model.predict_proba(test)[0][1]
prediction = "Annulation probable" if proba > 0.5 else "Reservation stable"

print("\n--- Test simulation formulaire CAMPINO ---")
print(f"Resultat    : {prediction}")
print(f"Score risque: {round(proba * 100, 1)}%")

Entrainement en cours...
Rapport de classification:
              precision    recall  f1-score   support

  Non annule       0.76      0.90      0.82     14973
      Annule       0.75      0.52      0.62      8905

    accuracy                           0.76     23878
   macro avg       0.76      0.71      0.72     23878
weighted avg       0.76      0.76      0.75     23878

ROC-AUC Score: 0.789
Modele sauvegarde : model.pkl

--- Test simulation formulaire CAMPINO ---
Resultat    : Annulation probable
Score risque: 54.0%
